In [1]:
%load_ext tensorboard

In [6]:
%tensorboard --logdir logs

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Tue Nov 28 15:11:58 2023

@author: teemu
"""

import numpy as np
import pandas as pd
import os
import glob
from copy import copy
import importlib.util
import sys

os.chdir('../')

dsc_spec = importlib.util.spec_from_file_location('dataset_collections', 'datasets/cancer_dataset_collections.py')
dsc = importlib.util.module_from_spec(dsc_spec)
sys.modules['dataset_collections'] = dsc
dsc_spec.loader.exec_module(dsc)

dsp_spec = importlib.util.spec_from_file_location('dataset_processing', 'datasets/cancer_dataset_processing.py')
dsp = importlib.util.module_from_spec(dsp_spec)
sys.modules['dataset_processing'] = dsp
dsp_spec.loader.exec_module(dsp)

et_spec = importlib.util.spec_from_file_location('evaluation_tools', 'utilities/evaluation_tools.py')
et = importlib.util.module_from_spec(et_spec)
sys.modules['evaluation_tools'] = et
et_spec.loader.exec_module(et)

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

pd.set_option('display.max_columns', None)

#%% Initialize tensorflow
import tensorflow as tf

gpu_memory = None
nthreads_interop = 80
nthreads = 2

gpus = tf.config.list_physical_devices('GPU')
if len(gpus) and gpu_memory is not None:
    print('Found {} GPUs, using 1 logical GPU with {} MB memory.'.format(len(gpus), gpu_memory))
    '''Fixed allocation, less overhead and easy to calculate'''
    tf.config.set_logical_device_configuration(
            gpus[0], # assume only one gpu
            [tf.config.LogicalDeviceConfiguration(memory_limit = gpu_memory)]
    )
    logical_gpus = tf.config.list_logical_devices('GPU')
else:
    pass
    #tf.config.threading.set_inter_op_parallelism_threads(nthreads_interop)
    #tf.config.threading.set_intra_op_parallelism_threads(nthreads)

#%% Parameters
from sys import platform
from socket import gethostname
if gethostname() == 'teemu-pc':
    base_path = '/home/teemu/research_work/superAE_HPO/'
    home = True
elif platform == 'linux':
    base_path = '/research/work/rintala/superAE_HPO/'
    home = False
else:
    base_path = '//research/workdir/superAE_HPO/'
    home = False

#res_path = base_path + '20230821_random_search/brca_test_noclfilter/'
#res_path = base_path + '20231027_random_search/brca_test_noclfilter/'
#res_path = base_path + '20231101_random_search/brca_test_noclfilter/'
#res_path = base_path + '20231103_random_search/brca_test_noclfilter/'
#res_path = base_path + '20231120_random_search/brca_test_noclfilter/'
#res_path = base_path + '20231121_random_search/brca_test_noclfilter/'
#res_path = base_path + '20231124_random_search/pancan_test/'
#res_path = base_path + '20240103_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240108_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240115_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240118_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240119_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240123_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240124_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240124_random_search/brca_test_noclfilter_alternative/'
#res_path = base_path + '20240125_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240125_random_search/brca_test_noclfilter_nopre/'
#res_path = base_path + '20240206_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240208_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240211_random_search/brca_test_noclfilter/'
#res_path = base_path + '20240213_random_search/brca_test_noclfilter/'
res_path = base_path + '20240213_random_search/brca_test_noclfilter_pretrain/'

# Should select by name
fn = glob.glob(res_path + '../plots/*best_task.txt')
with open(fn[0], 'r') as f:
    best_task = int(f.readlines()[0])

best_parameter_file = glob.glob(f"{res_path}*parameters_task{best_task}.csv")[0]

from dataset_collections import get_tcga_brca_ccle_full

data_dict = get_tcga_brca_ccle_full(home = home, gene_preselection = True)

if False:
    from dataset_collections import get_tcga_pancan_ccle_solid
    data_dict_full = get_tcga_pancan_ccle_solid(home = home)
    fn = res_path + '../plots/pan_cancer_ids.txt'
    with open(fn, 'w') as f:
        f.writelines(np.char.add(data_dict_full['patient_rows'], '\n'))
    for k, v in zip(*np.unique(data_dict_full['patient_cancer_type'], return_counts = True)):
        print(f"{k} : {v}")
    for k, v in zip(*np.unique(data_dict_full['cl_cancer_type'], return_counts = True)):
        print(f"{k} : {v}")

from evaluation_tools import get_kwargs

if False:
    data_dict_original = copy(data_dict)
    # Reduce dataset to CV size
    N_patient = data_dict['patient_exp'].shape[0]
    N_cv = int(np.floor(N_patient / 5 * 4))
    data_dict['patient_exp'] = data_dict['patient_exp'][:N_cv,:]
    data_dict['patient_rows'] = data_dict['patient_rows'][:N_cv]
    data_dict['survival_time'] = data_dict['survival_time'][:N_cv]
    data_dict['survival_event'] = data_dict['survival_event'][:N_cv]
    data_dict['survival_covariates'] = data_dict['survival_covariates'][:N_cv,:]
    data_dict['survival_mask'] = data_dict['survival_mask'][:N_cv]

search_kwargs = get_kwargs(
    parameter_file = best_parameter_file, 
    data_dict = data_dict)

#search_kwargs['data_standardize'] = False

#%% Serialization
from sae.data_utilities import JSONFeatureSpecDecoder
import json, re
from dataset_processing import process_and_serialize

if gethostname() == 'teemu-pc':
    tcga_ccle_serialized_data_path = '/home/teemu/Documents/superAE_temp/external_evaluation/tcga_ccl/'
elif platform == 'linux':
    tcga_ccle_serialized_data_path = f"{res_path}external_evaluation/tcga_ccl/"
else:
    tcga_ccle_serialized_data_path = f"{res_path}external_evaluation/tcga_ccl/"

spec_file = f"{tcga_ccle_serialized_data_path}serialized_data_spec.json"
if False and os.path.exists(spec_file):
    handle = open(spec_file, 'r')
    serialized_data_spec = json.load(handle)
    handle.close()
else:
    os.makedirs(tcga_ccle_serialized_data_path, exist_ok = True)
    model_args = search_kwargs['model_args']
    serialized_data_spec = process_and_serialize(
        model_args = model_args, 
        data_dict = data_dict, 
        omics_layer = search_kwargs['omics_layer'], 
        data_standardize = search_kwargs['data_standardize'], 
        serialized_data_path = tcga_ccle_serialized_data_path)

serialized_data = JSONFeatureSpecDecoder(serialized_data_spec)

if False:
    #search_kwargs['file_name_prefix'] = res_path
    spec_fn = glob.glob(res_path + '*_serialized_data_spec.json')[0]
    handle = open(spec_fn, 'r')
    serialized_data_cv = json.load(handle)
    handle.close()
    
    serialized_data_cv = JSONFeatureSpecDecoder(serialized_data_cv)
    
    serialized_data = {
        'patient' : serialized_data_cv['0']['1']['test_cv_']['patient']['train'], 
        'cl' : serialized_data_cv['0']['1']['test_cv_']['cell_line']['train']
    }
    
    serialized_data['patient']['filename'] = re.sub(
        '.*/', res_path, serialized_data['patient']['filename'])
    serialized_data['cl']['filename'] = re.sub(
        '.*/', res_path, serialized_data['cl']['filename'])

#%% Setup training
model_args = search_kwargs['model_args']

np.random.seed(0)
if model_args.get('encoder_layers', None):
    model_args['encoder_init_seeds'] = np.random.randint(2**31, size = len(model_args.get('encoder_layers')), dtype=np.int32)
if model_args.get('decoder_layers', None):
    model_args['decoder_init_seeds'] = np.random.randint(2**31, size = len(model_args.get('decoder_layers')), dtype=np.int32)
model_args['recon_init_seed'] = np.random.randint(2**31, size = 1, dtype=np.int32)[0]
if model_args.get('classifier_layers', None):
    model_args['classifier_init_seeds'] = np.random.randint(2**31, size = len(model_args.get('classifier_layers')), dtype=np.int32)
model_args['classifier_final_init_seed'] = np.random.randint(2**31, size = 1, dtype=np.int32)[0]
if model_args.get('survival_model_layers', None):
    model_args['survival_model_init_seeds'] = np.random.randint(2**31, size = len(model_args.get('survival_model_layers')), dtype=np.int32)
model_args['survival_model_final_init_seed'] = np.random.randint(2**31, size = 1, dtype=np.int32)[0]
if model_args.get('batch_adversarial_model_layers', None):
    model_args['batch_adversarial_model_init_seeds'] = np.random.randint(2**31, size = len(model_args.get('batch_adversarial_model_layers')), dtype=np.int32)
model_args['batch_adversarial_model_pred_seed'] = np.random.randint(2**31, size = 1, dtype=np.int32)[0]
if model_args.get('drug_response_model_layers', None):
    model_args['drug_response_model_init_seeds'] = np.random.randint(2**31, size = len(model_args.get('drug_response_model_layers')), dtype=np.int32)
model_args['drug_response_model_pred_seed'] = np.random.randint(2**31, size = 1, dtype=np.int32)[0]

#%% Load serialized data
from sae.data_utilities import parse_serialized_dataset_from_file

data_instance = serialized_data

tcga_patient_dataset = parse_serialized_dataset_from_file(
    serialized_file = data_instance['patient']['filename'],
    feature_spec = data_instance['patient']['feature_spec'])
tcga_patient_rows = np.array(data_instance['patient']['sample_info']['rownames'])
tcga_patient_model_spec = data_instance['patient']['model_spec']
tcga_patient_datasize = len(data_instance['patient']['sample_info']['rownames'])

ccle_cl_dataset = parse_serialized_dataset_from_file(
    serialized_file = data_instance['cl']['filename'],
    feature_spec = data_instance['cl']['feature_spec'])
ccle_cl_rows = np.array(data_instance['cl']['sample_info']['rownames'])
ccle_cl_model_spec = data_instance['cl']['model_spec']
ccle_cl_datasize = len(data_instance['cl']['sample_info']['rownames'])

cl_in = ccle_cl_model_spec.pop('input_dim')
if cl_in != tcga_patient_model_spec['input_dim']:
    raise ValueError('Input dimensions for serialized patient and cell-line data do not match.')

data_model_spec = {**tcga_patient_model_spec, **ccle_cl_model_spec}

#%% Training
from sae.training_utilities import train_aecl_with_pretraining

model_args = copy(search_kwargs['model_args'])
data_batch_args = copy(search_kwargs['data_batch_args'])
train_args = copy(search_kwargs['train_args'])
gym_args = copy(search_kwargs['gym_args'])


/tmp/ipykernel_432204/634837912.py:10: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
2024-02-15 17:44:37.730931: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-02-15 17:44:37.750879: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-15 17:44:37.750891: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugi

In [4]:

if True:
    # Shorter training for testing
    gym_args['max_epochs'] = 10
    gym_args['max_epochs_pre_ae'] = 10
    gym_args['max_epochs_pre_cl'] = 10
    gym_args['max_epochs_pre_sr'] = 10
    gym_args['max_epochs_pre_bd'] = 10
    gym_args['max_epochs_pre_bc'] = 10
    gym_args['max_epochs_pre_dr'] = 10
    
    #gym_args['pre_train_ae'] = False
    #gym_args['pre_train_bd'] = False
    
    model_args['deconfounder_norm_penalty'] = 1.
    #model_args['deconfounder_centered_alignment'] = True
    # Test variational survival
    #model_args['survival_variational'] = False
    #model_args['variational'] = True
    
    data_batch_args['batch_size'] = 128
    #data_batch_args['prefetch'] = True
    train_args['return_losses'] = False


In [5]:

tf.profiler.experimental.start('logs')
result = train_aecl_with_pretraining(
    patient_serialized_dataset = tcga_patient_dataset, 
    cl_serialized_dataset = ccle_cl_dataset, 
    model_args = {**model_args, **data_model_spec},
    data_batch_args = data_batch_args, 
    train_args = train_args, 
    patient_datasize = tcga_patient_datasize,
    cl_datasize = ccle_cl_datasize, 
    **gym_args, 
    return_pre_trained_model_weights = True)
tf.profiler.experimental.stop()


2024-02-15 17:46:09.194804: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:104] Profiler session initializing.
2024-02-15 17:46:09.194822: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:119] Profiler session started.
2024-02-15 17:46:31.941986: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:70] Profiler session collecting data.
2024-02-15 17:46:32.773113: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:131] Profiler session tear down.


In [ ]:

if False:
    model_final_weights = result['model'].get_weights()
    [(i.shape, np.min(i), np.max(i)) for i in result['model_bc_pre_weights']]
    result['train_losses_ae']
    result['valid_losses_ae']
    result['train_losses_bd']
    result['valid_losses_bd']
    result['train_losses_bc']
    result['valid_losses_bc']
    result['train_losses_sr']
    result['valid_losses_sr']
    result['train_losses_dr']
    result['valid_losses_dr']

data_batch_args['repeat'] = False

#%% Diagnostics
diagnostics_keys = [
    'train_losses_ae', 
    'valid_losses_ae',
    'train_losses_cl',
    'valid_losses_cl', 
    'train_losses_sr',
    'valid_losses_sr', 
    'train_losses_dr',
    'valid_losses_dr', 
    'train_losses_bd',
    'valid_losses_bd', 
    'train_losses_bc',
    'valid_losses_bc', 
    'train_losses', 
    'valid_losses']
diagnostics = [result[i] for i in diagnostics_keys if result.get(i, None) is not None]
diagnostics_stage = [i for i in diagnostics_keys if result.get(i, None) is not None]
for d, s in zip(diagnostics, diagnostics_stage):
    d['stage'] = s
if len(diagnostics) > 0:
    diagnostics = pd.concat(diagnostics, axis = 0)
    diagnostics.to_csv(
        f"{res_path}external_evaluation/diagnostics.csv.gz", 
        na_rep = 'NA', 
        header = True, 
        index = False)

#%% Evaluation
from sae.data_utilities import dataset_batch_setup
from sae.evaluation import get_model_losses

tcga_patient_dataset_batched = dataset_batch_setup(tcga_patient_dataset, **data_batch_args)

tcga_res = get_model_losses(
    result['model'], 
    dataset = tcga_patient_dataset_batched, 
    get_metrics = True, 
    tuple_dataset = False, 
    manual_batch_flag = False)

#%% Predictions and embeddings 

from sae.evaluation import get_embeddings, get_model_predictions

ccle_cl_dataset_batched = dataset_batch_setup(ccle_cl_dataset, **data_batch_args)

model = result['model']

datasets = [
    {
        'dataset' : tcga_patient_dataset_batched, 
        'rows' : tcga_patient_rows, 
        'name' : 'tcga', 
        'file_string' : f"{res_path}external_evaluation/internal_survival_validation_"
    },
    {
        'dataset' : ccle_cl_dataset_batched, 
        'rows' : ccle_cl_rows, 
        'name' : 'ccle', 
        'file_string' : f"{res_path}external_evaluation/internal_drug_response_validation_"
    }
]

dataset_rows = [i['rows'] for i in datasets if i is not None]
dataset_names = [i['name'] for i in datasets if i is not None]
dataset_files = [i['file_string'] for i in datasets if i is not None]
datasets = [i['dataset'] for i in datasets if i is not None]

phases_of_interest = ['final', 'ae_pt', 'bc_pt']
weight_key_list = ['model_ae_pre_weights', 'model_cl_pre_weights', 
                   'model_sr_pre_weights', 'model_bd_pre_weights', 
                   'model_bc_pre_weights', 'model_dr_pre_weights']
trained_list = [True]
weight_list = [copy(model.get_weights())]
weight_names = ['final']

trained_list += [result.get(i, None) is not None for i in weight_key_list]
weight_list += [result.get(i, None) for i in weight_key_list]
weight_names += ['ae_pt', 'cl_pt', 'sr_pt', 'bd_pt', 'bc_pt', 'dr_pt']

prediction_model = (model.supervised or model.survival_model or model.drug_response_model)
for trained, weights, weight_name in zip(trained_list, weight_list, weight_names):
    if trained and weight_name in phases_of_interest:
        model.set_weights(weights)
        for dataset, rows, dataset_name, file_prefix in zip(datasets, dataset_rows, dataset_names, dataset_files):
            zi = get_embeddings(model, dataset, tuple_dataset = False)
            zi = pd.DataFrame(
                zi, 
                index = rows,
                columns = ['z{}'.format(i+1) for i in range(zi.shape[1])])
            zi['dataset'] = dataset_name
            zi.to_csv(
                f"{file_prefix}{weight_name}_embeddings.csv.gz", 
                na_rep = 'NA', 
                header = True, 
                index = True)
            if weight_name == 'final' and prediction_model:
                pi = get_model_predictions(model, dataset, tuple_dataset = False)
                for pred, key in zip(pi.values(), pi.keys()):
                    if pred.shape[1]:
                        cols = [key + '_' + str(i) for i in np.arange(pred.shape[1])]
                    else:
                        cols = key
                    pi[key] = pd.DataFrame(pi[key], index = rows, columns = cols)
                pi = pd.concat(pi.values(), axis = 1)
                pi['dataset'] = dataset_name
                pi.to_csv(
                    f"{file_prefix}{weight_name}_predictions.csv.gz", 
                    na_rep = 'NA', 
                    header = True, 
                    index = True)

model.set_weights(weight_list[0]) # Set back to final weights

#%% SCAN-B dataset new
from dataset_collections import get_scanb

scanb_data_dict = get_scanb(home = home)

# Filter to common genes
scanb_original_expression = scanb_data_dict['patient_exp']

reordering_map = dict(zip(data_dict['gene_ids'], np.arange(data_dict['gene_ids'].shape[0])))
new_scanb_gex_mat = np.full(
    (scanb_data_dict['patient_exp'].shape[0], 
    data_dict['gene_ids'].shape[0]), 0.)
for i,g in enumerate(scanb_data_dict['gene_ids']):
    tcga_ind = reordering_map.get(g, None)
    if tcga_ind:
        new_scanb_gex_mat[:,tcga_ind] = scanb_data_dict['patient_exp'][:,i]
#new_scanb_gex_mat[]
scanb_data_dict['patient_exp'] = new_scanb_gex_mat# * 0.

#%% Serialize SCAN-B
scanb_serialized_data_path = tcga_ccle_serialized_data_path + '../scanb/'
scanb_spec_file = f"{scanb_serialized_data_path}serialized_data_spec.json"
if False and os.path.exists(scanb_spec_file):
    handle = open(scanb_spec_file, 'r')
    scanb_serialized_data_spec = json.load(handle)
    handle.close()
else:
    os.makedirs(scanb_serialized_data_path, exist_ok = True)
    model_args = search_kwargs['model_args']
    scanb_serialized_data_spec = process_and_serialize(
        model_args = model_args, 
        data_dict = scanb_data_dict, 
        omics_layer = search_kwargs['omics_layer'], 
        data_standardize = search_kwargs['data_standardize'], 
        serialized_data_path = scanb_serialized_data_path)

serialized_external_data = JSONFeatureSpecDecoder(scanb_serialized_data_spec)

#%% Load serialized data
external_data_instance = serialized_external_data

scanb_patient_dataset = parse_serialized_dataset_from_file(
    serialized_file = external_data_instance['patient']['filename'],
    feature_spec = external_data_instance['patient']['feature_spec'])
scanb_patient_rows = np.array(external_data_instance['patient']['sample_info']['rownames'])
scanb_patient_model_spec = external_data_instance['patient']['model_spec']
scanb_patient_datasize = len(external_data_instance['patient']['sample_info']['rownames'])

#%% SCANB Evaluation
scanb_patient_dataset_batched = dataset_batch_setup(scanb_patient_dataset, **data_batch_args)

scanb_res = get_model_losses(
    result['model'], 
    dataset = scanb_patient_dataset_batched, 
    get_metrics = True, 
    tuple_dataset = False, 
    manual_batch_flag = False)

#%% SCANB Predictions and embeddings for patients

from sae.evaluation import get_embeddings, get_model_predictions

z = get_embeddings(result['model'], scanb_patient_dataset_batched, tuple_dataset = False)
z = pd.DataFrame(
    z, 
    index = scanb_patient_rows,
    columns = ['z{}'.format(i+1) for i in range(z.shape[1])])
z['dataset'] = 'scanb'

p = get_model_predictions(result['model'], scanb_patient_dataset_batched, tuple_dataset = False)
for pred, key in zip(p.values(), p.keys()):
    if pred.shape[1]:
        cols = [key + '_' + str(i) for i in np.arange(pred.shape[1])]
    else:
        cols = key
    p[key] = pd.DataFrame(p[key], index = scanb_patient_rows, columns = cols)
p = pd.concat(p.values(), axis = 1)
p['dataset'] = 'scanb'

z.to_csv(
    f"{res_path}external_evaluation/external_survival_validation_embeddings.csv.gz", 
    na_rep = 'NA', 
    header = True, 
    index = True)
p.to_csv(
    f"{res_path}external_evaluation/external_survival_validation_predictions.csv.gz", 
    na_rep = 'NA', 
    header = True, 
    index = True)

#%% Save results

result_df = pd.DataFrame({
    'dataset' : ('TCGA_BRCA', 'SCANB'),
    'reconstruction_mse' : (tcga_res['reconstruction_loss_dataset1'], 
                            scanb_res['reconstruction_loss_dataset1']), 
    'regularization' : (tcga_res['regularization'], 
                        scanb_res['regularization']), 
    'survival_log_likelihood' : (tcga_res['survival_log_likelihood'], 
                                 scanb_res['survival_log_likelihood']), 
    'survival_concordance' : (tcga_res['surv_c'], 
                              scanb_res['surv_c']), 
    'confounder_alignment_norm' : (tcga_res['confounder_alignment_norm'], 
                                   scanb_res['confounder_alignment_norm']) 
    })
result_df.to_csv(f"{res_path}external_evaluation/external_survival_validation_res.csv")

#%% Bruna PDTC and PDTX gene-expression datasets
from dataset_collections import get_bruna_pdtc, get_bruna_pdtx

pdtc_data_dict = get_bruna_pdtc(home = home)
pdtx_data_dict = get_bruna_pdtx(home = home)

# Filter to common genes
for dd in [pdtc_data_dict, pdtx_data_dict]:
    og_expression = dd['cl_exp']
    
    reordering_map = dict(zip(data_dict['gene_ids'], np.arange(data_dict['gene_ids'].shape[0])))
    new_gex_mat = np.full(
        (dd['cl_exp'].shape[0], 
        data_dict['gene_ids'].shape[0]), 0.)
    for i,g in enumerate(dd['gene_ids']):
        tcga_ind = reordering_map.get(g, None)
        if tcga_ind:
            new_gex_mat[:,tcga_ind] = dd['cl_exp'][:,i]
    dd['cl_exp'] = new_gex_mat# * 0.

#%% Serialize external datasets

serialized_ext_data_list = []
for dd, sub_path in zip([pdtc_data_dict, pdtx_data_dict], ['bruna_pdtc/', 'bruna_pdtx/']):
    serialized_ext_data_path = tcga_ccle_serialized_data_path + '../' + sub_path
    ext_spec_file = f"{serialized_ext_data_path}serialized_data_spec.json"
    if False and os.path.exists(ext_spec_file):
        handle = open(ext_spec_file, 'r')
        serialized_ext_data_spec = json.load(handle)
        handle.close()
    else:
        os.makedirs(serialized_ext_data_path, exist_ok = True)
        model_args = search_kwargs['model_args']
        serialized_ext_data_spec = process_and_serialize(
            model_args = model_args, 
            data_dict = dd, 
            omics_layer = search_kwargs['omics_layer'], 
            data_standardize = search_kwargs['data_standardize'], 
            serialized_data_path = serialized_ext_data_path)
    
    serialized_ext_data_list.append(JSONFeatureSpecDecoder(serialized_ext_data_spec))

#%% Load serialized data
embeddings = []
predictions = []
for edi, name in zip(serialized_ext_data_list, ['bruna_pdtc', 'bruna_pdtx']):
    ext_dataset = parse_serialized_dataset_from_file(
        serialized_file = edi['cl']['filename'],
        feature_spec = edi['cl']['feature_spec'])
    ext_rows = np.array(edi['cl']['sample_info']['rownames'])
    ext_model_spec = edi['cl']['model_spec']
    ext_datasize = len(edi['cl']['sample_info']['rownames'])
    
    ext_dataset_batched = dataset_batch_setup(ext_dataset, **data_batch_args)
    
    z = get_embeddings(result['model'], ext_dataset_batched, tuple_dataset = False)
    z = pd.DataFrame(
        z, 
        index = ext_rows,
        columns = ['z{}'.format(i+1) for i in range(z.shape[1])])
    z['dataset'] = name
    embeddings.append(z)
    
    p = get_model_predictions(result['model'], ext_dataset_batched, tuple_dataset = False)
    for pred, key in zip(p.values(), p.keys()):
        if pred.shape[1]:
            cols = [key + '_' + str(i) for i in np.arange(pred.shape[1])]
        else:
            cols = key
        p[key] = pd.DataFrame(p[key], index = ext_rows, columns = cols)
    p = pd.concat(p.values(), axis = 1)
    p['dataset'] = name
    predictions.append(p)

embeddings = pd.concat(embeddings, axis = 0)
predictions = pd.concat(predictions, axis = 0)
embeddings.to_csv(
    f"{res_path}external_evaluation/external_drug_response_validation_embeddings.csv.gz", 
    na_rep = 'NA', 
    header = True, 
    index = True)
predictions.to_csv(
    f"{res_path}external_evaluation/external_drug_response_validation_predictions.csv.gz", 
    na_rep = 'NA', 
    header = True, 
    index = True)